# 185. 把 Agent Loop 建模为 POMDP：Belief State、主动观察与安全行动怎样实现？

> **面试问题：为什么 observation 不等于真实 state？Agent 怎样用 noisy tool result 更新 belief，并在高风险动作前主动取证？**

## 先给结论

真实网页、GUI 和工具只暴露部分且可能过期的 observation。POMDP 视角让 Agent 维护对隐藏 state 的 belief，经 observation likelihood 做 Bayesian update，再按期望效用和信息价值选 action。高风险动作不应由单次模型猜测触发；应设置信念阈值、主动观察、状态新鲜度和执行后验证。

## 推荐回答主线

1. 定义隐藏 state、observation、action、transition/reward 与 belief，区分原始历史和充分统计量。
2. 实现 Bayesian belief update，验证归一化、证据顺序和不可能 observation 的失败策略。
3. 比较 inspect/execute/abort 的期望效用与信息增益，高风险动作要求低危险概率和新鲜证据。
4. 把 belief/observation 版本写入 trace；用成功、损害、观察成本、校准和恢复能力评估。

## 教学实现边界

二状态玩具环境的 likelihood 已知；真实 Agent 的状态空间和 observation model 通常未知，只能近似。公式用于组织安全决策，不声称 LLM 能精确做 Bayesian inference。

## 一手资料

- [OSWorld](https://arxiv.org/abs/2404.07972)
- [WebArena](https://arxiv.org/abs/2307.13854)
- [AgentBench](https://arxiv.org/abs/2308.03688)


In [ ]:
import hashlib  # 导入本单元需要的依赖。
import json  # 导入本单元需要的依赖。
import math  # 导入本单元需要的依赖。
from dataclasses import asdict, dataclass  # 导入本单元需要的依赖。

import numpy as np  # 导入本单元需要的依赖。

# 隐藏状态 SAFE/DANGER；工具只返回带版本和时间的有噪声 observation。
STATES = ("safe", "danger")  # 计算并保存当前步骤的中间状态。
OBSERVATIONS = ("green", "red")  # 计算并保存当前步骤的中间状态。
LIKELIHOOD = {  # 计算并保存当前步骤的中间状态。
    "safe": {"green": 0.9, "red": 0.1},  # 执行当前语句以推进本节示例。
    "danger": {"green": 0.2, "red": 0.8},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class ObservationRecord:  # 定义承载本节状态与行为的数据结构。
    value: str  # 执行当前语句以推进本节示例。
    source: str  # 执行当前语句以推进本节示例。
    state_version: int  # 执行当前语句以推进本节示例。
    observed_at: int  # 执行当前语句以推进本节示例。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class POMDPArtifact:  # 定义承载本节状态与行为的数据结构。
    environment: str  # 执行当前语句以推进本节示例。
    observation_model: str  # 执行当前语句以推进本节示例。
    action_policy: str  # 执行当前语句以推进本节示例。
    danger_threshold: float  # 执行当前语句以推进本节示例。
    freshness_limit: int  # 执行当前语句以推进本节示例。
    inspect_cost: float  # 执行当前语句以推进本节示例。

belief = np.array([0.6, 0.4], dtype=float)  # 计算并保存当前步骤的中间状态。

assert math.isclose(belief.sum(), 1.0)  # 用受控断言验证关键不变量。
assert set(LIKELIHOOD) == set(STATES)  # 用受控断言验证关键不变量。
assert all(math.isclose(sum(row.values()), 1.0) for row in LIKELIHOOD.values())  # 用受控断言验证关键不变量。


## 1. Bayesian update：Observation 是证据，不是 state 标签

posterior 与 prior×likelihood 成正比。一次 green 不能把 danger 概率设成 0；工具也可能错。归一化常数为零时应拒绝更新并记录 model mismatch。


In [ ]:
def validate_belief(prior):  # 定义本节可复用的核心函数。
    array = np.asarray(prior, dtype=float)  # 计算并保存当前步骤的中间状态。
    if array.shape != (len(STATES),) or not np.isfinite(array).all() or (array < 0).any():  # 按当前条件选择后续控制路径。
        raise ValueError("prior 必须是有限、非负且与状态数一致的向量")  # 遇到非法合同立即显式失败。
    if not math.isclose(float(array.sum()), 1.0, rel_tol=0.0, abs_tol=1e-9):  # 按当前条件选择后续控制路径。
        raise ValueError("prior 概率和必须为 1")  # 遇到非法合同立即显式失败。
    return array.copy()  # 返回当前分支计算出的结果。

def validate_likelihood(likelihood):  # 定义本节可复用的核心函数。
    if set(likelihood) != set(STATES):  # 按当前条件选择后续控制路径。
        raise ValueError("likelihood 状态集合错误")  # 遇到非法合同立即显式失败。
    for state in STATES:  # 遍历输入元素以累积或检查结果。
        row = likelihood[state]  # 计算并保存当前步骤的中间状态。
        if set(row) != set(OBSERVATIONS):  # 按当前条件选择后续控制路径。
            raise ValueError("likelihood observation 集合错误")  # 遇到非法合同立即显式失败。
        values = np.array([row[item] for item in OBSERVATIONS], dtype=float)  # 计算并保存当前步骤的中间状态。
        if not np.isfinite(values).all() or (values < 0).any() or not math.isclose(float(values.sum()), 1.0, abs_tol=1e-9):  # 按当前条件选择后续控制路径。
            raise ValueError("likelihood 每行必须是有限概率分布")  # 遇到非法合同立即显式失败。
    return likelihood  # 返回当前分支计算出的结果。

def validate_observation_record(record):  # 定义本节可复用的核心函数。
    if not isinstance(record, ObservationRecord):  # 按当前条件选择后续控制路径。
        raise TypeError("observation_stream 只能产生 ObservationRecord")  # 遇到非法合同立即显式失败。
    if record.value not in OBSERVATIONS or not isinstance(record.source, str) or not record.source:  # 按当前条件选择后续控制路径。
        raise ValueError("observation value/source 非法")  # 遇到非法合同立即显式失败。
    if type(record.state_version) is not int or type(record.observed_at) is not int or min(record.state_version, record.observed_at) < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("observation version/time 必须是非负整数")  # 遇到非法合同立即显式失败。
    return record  # 返回当前分支计算出的结果。

def observation_status(record, current_version, now, max_age):  # 定义本节可复用的核心函数。
    validate_observation_record(record)  # 执行当前语句以推进本节示例。
    if any(type(value) is not int for value in (current_version, now, max_age)) or min(current_version, now, max_age) < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("current_version/now/max_age 必须是非负整数")  # 遇到非法合同立即显式失败。
    if record.state_version != current_version:  # 按当前条件选择后续控制路径。
        return "wrong_version"  # 返回当前分支计算出的结果。
    age = now - record.observed_at  # 计算并保存当前步骤的中间状态。
    if age < 0:  # 按当前条件选择后续控制路径。
        return "future"  # 返回当前分支计算出的结果。
    if age > max_age:  # 按当前条件选择后续控制路径。
        return "stale"  # 返回当前分支计算出的结果。
    return "accepted"  # 返回当前分支计算出的结果。

def version_matches(record, current_version, now, max_age):  # 定义本节可复用的核心函数。
    return observation_status(record, current_version, now, max_age) == "accepted"  # 返回当前分支计算出的结果。

def update_belief(prior, observation, likelihood=LIKELIHOOD):  # 定义本节可复用的核心函数。
    prior = validate_belief(prior)  # 计算并保存当前步骤的中间状态。
    validate_likelihood(likelihood)  # 执行当前语句以推进本节示例。
    if observation not in OBSERVATIONS:  # 按当前条件选择后续控制路径。
        raise ValueError("未知 observation")  # 遇到非法合同立即显式失败。
    evidence = np.array([likelihood[state][observation] for state in STATES], dtype=float)  # 计算并保存当前步骤的中间状态。
    unnormalized = prior * evidence  # 计算并保存当前步骤的中间状态。
    if not np.isfinite(unnormalized).all() or unnormalized.sum() <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("observation 在模型下不可能")  # 遇到非法合同立即显式失败。
    return unnormalized / unnormalized.sum()  # 返回当前分支计算出的结果。

# green/red 方向正确；非法 prior 与未知 observation 均 fail closed。
after_green = update_belief(belief, "green")  # 计算并保存当前步骤的中间状态。
after_red = update_belief(belief, "red")  # 计算并保存当前步骤的中间状态。
assert 0 < after_green[1] < belief[1] < after_red[1]  # 用受控断言验证关键不变量。
assert math.isclose(after_green.sum(), 1.0)  # 用受控断言验证关键不变量。
for invalid_prior in (np.array([float("nan"), 0.0]), np.array([-0.1, 1.1]), np.array([0.2, 0.2])):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        update_belief(invalid_prior, "green"); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 2. 连续观察：条件独立假设若不成立会过度自信

在给定 state 条件独立的简化下可顺序更新；若两次结果来自同一缓存/同一模型，它们高度相关，不能当独立证据重复乘 likelihood。provenance 要标 observation source 与 freshness。


In [ ]:
def apply_observations(prior, observations, likelihood=LIKELIHOOD):  # 定义本节可复用的核心函数。
    posterior = validate_belief(prior)  # 计算并保存当前步骤的中间状态。
    for observation in observations:  # 遍历输入元素以累积或检查结果。
        posterior = update_belief(posterior, observation, likelihood)  # 计算并保存当前步骤的中间状态。
    return posterior  # 返回当前分支计算出的结果。

# 独立观测可顺序更新；未知观测不能被当作无害空值跳过。
two_green = apply_observations(belief, ["green", "green"])  # 计算并保存当前步骤的中间状态。
mixed_a = apply_observations(belief, ["green", "red"])  # 计算并保存当前步骤的中间状态。
mixed_b = apply_observations(belief, ["red", "green"])  # 计算并保存当前步骤的中间状态。
assert two_green[1] < after_green[1]  # 用受控断言验证关键不变量。
assert np.allclose(mixed_a, mixed_b)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    apply_observations(belief, ["blue"]); assert False  # 执行当前语句以推进本节示例。
except ValueError:  # 捕获预期异常并验证失败分支。
    assert True  # 用受控断言验证关键不变量。


## 3. 期望效用：execute 的收益与损害不对称

安全状态执行收益 +10，危险状态执行损失 -50，abort 为 0。即使 safe 概率过半，期望效用仍可能为负。风险偏好和不可逆损害应由产品 policy 设定，不交给 prompt 临时决定。


In [ ]:
UTILITY = {"execute": np.array([10.0, -50.0]), "abort": np.array([0.0, 0.0])}  # 计算并保存当前步骤的中间状态。

def expected_utility(action, current_belief):  # 定义本节可复用的核心函数。
    if action not in UTILITY:  # 按当前条件选择后续控制路径。
        raise ValueError("未知 action")  # 遇到非法合同立即显式失败。
    return float(validate_belief(current_belief) @ UTILITY[action])  # 返回当前分支计算出的结果。

# 初始执行期望为负；green 后改善；abort 恒为零。
assert expected_utility("execute", belief) < 0  # 用受控断言验证关键不变量。
assert expected_utility("execute", after_green) > expected_utility("execute", belief)  # 用受控断言验证关键不变量。
assert expected_utility("abort", after_red) == 0  # 用受控断言验证关键不变量。


## 4. 信息增益：Inspect 的价值是改变后续决策，不是多调用工具

belief entropy 衡量不确定性。inspect 的期望信息增益是观察前熵减去各 observation posterior 熵的期望，再减调用成本。若无论观察什么都不会改变动作，继续 inspect 可能浪费预算。


In [ ]:
def belief_entropy(probabilities):  # 定义本节可复用的核心函数。
    p = validate_belief(probabilities)  # 计算并保存当前步骤的中间状态。
    positive = p > 0  # 计算并保存当前步骤的中间状态。
    return float(-(p[positive] * np.log(p[positive])).sum())  # 返回当前分支计算出的结果。

def expected_information_gain(prior, likelihood=LIKELIHOOD):  # 定义本节可复用的核心函数。
    prior = validate_belief(prior)  # 计算并保存当前步骤的中间状态。
    validate_likelihood(likelihood)  # 执行当前语句以推进本节示例。
    before, expected_after = belief_entropy(prior), 0.0  # 计算并保存当前步骤的中间状态。
    for observation in OBSERVATIONS:  # 遍历输入元素以累积或检查结果。
        p_observation = sum(prior[index] * likelihood[state][observation] for index, state in enumerate(STATES))  # 计算并保存当前步骤的中间状态。
        if p_observation > 0:  # 按当前条件选择后续控制路径。
            expected_after += p_observation * belief_entropy(update_belief(prior, observation, likelihood))  # 计算并保存当前步骤的中间状态。
    return max(0.0, before - expected_after)  # 返回当前分支计算出的结果。

UNINFORMATIVE_LIKELIHOOD = {  # 计算并保存当前步骤的中间状态。
    "safe": {"green": 0.5, "red": 0.5},  # 执行当前语句以推进本节示例。
    "danger": {"green": 0.5, "red": 0.5},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

# 有信息传感器 EIG>0；无信息传感器和确定 belief 的 EIG 为零。
information_gain = expected_information_gain(belief)  # 计算并保存当前步骤的中间状态。
assert information_gain > 0  # 用受控断言验证关键不变量。
assert expected_information_gain(belief, UNINFORMATIVE_LIKELIHOOD) < 1e-12  # 用受控断言验证关键不变量。
assert expected_information_gain(np.array([1.0, 0.0])) < 1e-12  # 用受控断言验证关键不变量。


## 5. 主动观察策略：不确定或高风险时先 inspect

可设 danger 上限与 freshness 门禁：只有新鲜 posterior 且危险概率低于阈值才 execute；否则在预算内 inspect，耗尽后 abort/升级人工。阈值来自损害成本和校准，不是 0.5。


In [ ]:
def validate_policy(policy):  # 定义本节可复用的核心函数。
    if not isinstance(policy, POMDPArtifact):  # 按当前条件选择后续控制路径。
        raise TypeError("policy 必须是 POMDPArtifact")  # 遇到非法合同立即显式失败。
    if not all(isinstance(value, str) and value for value in (policy.environment, policy.observation_model, policy.action_policy)):  # 按当前条件选择后续控制路径。
        raise ValueError("artifact 版本字段不得为空")  # 遇到非法合同立即显式失败。
    if not isinstance(policy.danger_threshold, (int, float)) or not math.isfinite(policy.danger_threshold) or not 0 <= policy.danger_threshold <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError("danger_threshold 必须在 [0,1]")  # 遇到非法合同立即显式失败。
    if type(policy.freshness_limit) is not int or policy.freshness_limit < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("freshness_limit 必须是非负整数")  # 遇到非法合同立即显式失败。
    if not isinstance(policy.inspect_cost, (int, float)) or not math.isfinite(policy.inspect_cost) or policy.inspect_cost < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("inspect_cost 必须有限非负")  # 遇到非法合同立即显式失败。
    return policy  # 返回当前分支计算出的结果。

DEFAULT_POLICY = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.01)  # 计算并保存当前步骤的中间状态。

def choose_action(current_belief, evidence_count, observation_age, inspect_budget, policy=DEFAULT_POLICY, likelihood=LIKELIHOOD, return_diagnostics=False):  # 定义本节可复用的核心函数。
    current_belief = validate_belief(current_belief)  # 计算并保存当前步骤的中间状态。
    validate_policy(policy); validate_likelihood(likelihood)  # 执行当前语句以推进本节示例。
    if type(evidence_count) is not int or evidence_count < 0 or type(inspect_budget) is not int or inspect_budget < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("evidence_count/inspect_budget 必须是非负整数")  # 遇到非法合同立即显式失败。
    if evidence_count == 0:  # 按当前条件选择后续控制路径。
        if observation_age is not None:  # 按当前条件选择后续控制路径。
            raise ValueError("无有效证据时 observation_age 必须为 None")  # 遇到非法合同立即显式失败。
        fresh_evidence = False  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        if type(observation_age) is not int or observation_age < 0:  # 按当前条件选择后续控制路径。
            raise ValueError("observation_age 必须是非负整数")  # 遇到非法合同立即显式失败。
        fresh_evidence = observation_age <= policy.freshness_limit  # 计算并保存当前步骤的中间状态。
    danger = float(current_belief[1])  # 计算并保存当前步骤的中间状态。
    eig = expected_information_gain(current_belief, likelihood)  # 计算并保存当前步骤的中间状态。
    net_information_value = eig - policy.inspect_cost  # 计算并保存当前步骤的中间状态。
    can_execute = fresh_evidence and danger <= policy.danger_threshold and expected_utility("execute", current_belief) > 0  # 计算并保存当前步骤的中间状态。
    action = "execute" if can_execute else "inspect" if inspect_budget > 0 and net_information_value > 0 else "abort"  # 计算并保存当前步骤的中间状态。
    diagnostics = {"eig": eig, "inspect_cost": policy.inspect_cost, "net_information_value": net_information_value, "fresh_evidence": fresh_evidence}  # 计算并保存当前步骤的中间状态。
    return (action, diagnostics) if return_diagnostics else action  # 返回当前分支计算出的结果。

# 无证据不能 execute；EIG 扣成本后决定是否 inspect；无信息传感器不浪费预算。
assert choose_action(belief, 0, None, 2) == "inspect"  # 用受控断言验证关键不变量。
assert choose_action(two_green, 2, 0, 1) == "execute"  # 用受控断言验证关键不变量。
assert choose_action(np.array([1.0, 0.0]), 0, None, 2) == "abort"  # 用受控断言验证关键不变量。
assert choose_action(belief, 0, None, 2, likelihood=UNINFORMATIVE_LIKELIHOOD) == "abort"  # 用受控断言验证关键不变量。
for bad_budget in (-1, 1.5):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        choose_action(belief, 0, None, bad_budget); assert False  # 执行当前语句以推进本节示例。
    except ValueError:  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 6. Belief-state Agent Loop：Observe→Update→Decide→Act/Stop

循环记录每次 observation 的 source/version/time，更新 belief 后再决策。执行前可做 compare-and-swap/页面重截图，执行后验证状态；这里用预设观测序列演示有界终止。


In [ ]:
def run_belief_agent(prior, observation_stream, inspect_budget=3, current_version=0, now=0, policy=DEFAULT_POLICY, likelihood=LIKELIHOOD):  # 定义本节可复用的核心函数。
    posterior = validate_belief(prior)  # 计算并保存当前步骤的中间状态。
    validate_policy(policy); validate_likelihood(likelihood)  # 执行当前语句以推进本节示例。
    if type(inspect_budget) is not int or inspect_budget < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("inspect_budget 必须是非负整数")  # 遇到非法合同立即显式失败。
    if any(type(value) is not int for value in (current_version, now)) or min(current_version, now) < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("current_version/now 必须是非负整数")  # 遇到非法合同立即显式失败。
    try:  # 尝试执行可能失败的受控操作。
        stream = iter(observation_stream)  # 计算并保存当前步骤的中间状态。
    except TypeError as error:  # 捕获预期异常并验证失败分支。
        raise TypeError("observation_stream 必须可迭代") from error  # 遇到非法合同立即显式失败。
    trace, evidence_count, observation_age = [], 0, None  # 计算并保存当前步骤的中间状态。
    while True:  # 在终止条件满足前持续推进状态。
        action, diagnostics = choose_action(posterior, evidence_count, observation_age, inspect_budget, policy, likelihood, True)  # 计算并保存当前步骤的中间状态。
        trace.append({"state": "decide", "belief": posterior.copy(), "action": action, "evidence_count": evidence_count, "observation_age": observation_age, **diagnostics})  # 计算并保存当前步骤的中间状态。
        if action in {"execute", "abort"}:  # 按当前条件选择后续控制路径。
            return action, posterior, trace  # 返回当前分支计算出的结果。
        try:  # 尝试执行可能失败的受控操作。
            record = next(stream)  # 计算并保存当前步骤的中间状态。
        except StopIteration:  # 捕获预期异常并验证失败分支。
            trace.append({"state": "inspect", "status": "stream_exhausted"})  # 计算并保存当前步骤的中间状态。
            return "abort", posterior, trace  # 返回当前分支计算出的结果。
        inspect_budget -= 1  # 计算并保存当前步骤的中间状态。
        status = observation_status(record, current_version, now, policy.freshness_limit)  # 计算并保存当前步骤的中间状态。
        trace.append({"state": "observe", "value": record.value, "source": record.source, "state_version": record.state_version, "observed_at": record.observed_at, "status": status})  # 计算并保存当前步骤的中间状态。
        if status != "accepted":  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        posterior = update_belief(posterior, record.value, likelihood)  # 计算并保存当前步骤的中间状态。
        evidence_count += 1  # 计算并保存当前步骤的中间状态。
        observation_age = now - record.observed_at  # 计算并保存当前步骤的中间状态。

# 两条新鲜 green 经真实 loop 后执行；red 且预算耗尽则 abort。
good_records = [ObservationRecord("green", "sensor-a", 7, 100), ObservationRecord("green", "sensor-b", 7, 100)]  # 计算并保存当前步骤的中间状态。
bad_records = [ObservationRecord("red", "sensor-a", 7, 100)]  # 计算并保存当前步骤的中间状态。
action_good, final_good, trace_good = run_belief_agent(belief, good_records, 2, 7, 100)  # 计算并保存当前步骤的中间状态。
action_bad, _, trace_bad = run_belief_agent(belief, bad_records, 1, 7, 100)  # 计算并保存当前步骤的中间状态。
assert action_good == "execute" and final_good[1] < DEFAULT_POLICY.danger_threshold  # 用受控断言验证关键不变量。
assert action_bad == "abort"  # 用受控断言验证关键不变量。
assert sum(event["state"] == "observe" for event in trace_good) == 2  # 用受控断言验证关键不变量。
assert all(event.get("status") == "accepted" for event in trace_good if event["state"] == "observe")  # 用受控断言验证关键不变量。


## 7. 陈旧 observation 与状态漂移：时间也是隐藏状态

网页/文件可能在观察后被他人修改。belief 要绑定 snapshot/version；高风险动作使用 optimistic concurrency，版本不一致就重新观察。仅保留摘要而丢版本会制造 TOCTOU。


In [ ]:
# stale/future/wrong-version 都被主 loop 消费预算但不更新 belief，也绝不能促成 execute。
record = ObservationRecord("green", "sensor-a", 7, 100)  # 计算并保存当前步骤的中间状态。
assert version_matches(record, 7, 101, 2)  # 用受控断言验证关键不变量。
assert not version_matches(record, 8, 101, 2)  # 用受控断言验证关键不变量。
assert not version_matches(record, 7, 104, 2)  # 用受控断言验证关键不变量。
assert not version_matches(ObservationRecord("green", "sensor-a", 7, 102), 7, 101, 2)  # 用受控断言验证关键不变量。

safe_prior = np.array([0.95, 0.05])  # 计算并保存当前步骤的中间状态。
no_cost_policy = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.0)  # 计算并保存当前步骤的中间状态。
rejected_records = [  # 计算并保存当前步骤的中间状态。
    ObservationRecord("green", "sensor-a", 7, 90),  # 执行当前语句以推进本节示例。
    ObservationRecord("green", "sensor-a", 7, 102),  # 执行当前语句以推进本节示例。
    ObservationRecord("green", "sensor-a", 8, 101),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
expected_status = ["stale", "future", "wrong_version"]  # 计算并保存当前步骤的中间状态。
for invalid_record, status in zip(rejected_records, expected_status):  # 遍历输入元素以累积或检查结果。
    action, posterior, trace = run_belief_agent(safe_prior, [invalid_record], 1, 7, 101, no_cost_policy)  # 计算并保存当前步骤的中间状态。
    observations = [event for event in trace if event["state"] == "observe"]  # 计算并保存当前步骤的中间状态。
    assert action == "abort" and np.array_equal(posterior, safe_prior)  # 用受控断言验证关键不变量。
    assert [event["status"] for event in observations] == [status]  # 用受控断言验证关键不变量。
    assert not any(event.get("action") == "execute" for event in trace)  # 用受控断言验证关键不变量。

fresh_action, fresh_posterior, _ = run_belief_agent(safe_prior, [ObservationRecord("green", "sensor-a", 7, 101)], 1, 7, 101, no_cost_policy)  # 计算并保存当前步骤的中间状态。
assert fresh_action == "execute" and fresh_posterior[1] < safe_prior[1]  # 用受控断言验证关键不变量。
for bad_stream, bad_budget in (([ObservationRecord("blue", "sensor-a", 7, 101)], 1), ([], -1), ([], 1.5)):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        run_belief_agent(belief, bad_stream, bad_budget, 7, 101, no_cost_policy); assert False  # 执行当前语句以推进本节示例。
    except (TypeError, ValueError):  # 捕获预期异常并验证失败分支。
        assert True  # 用受控断言验证关键不变量。


## 8. 评测与制品：成功率之外必须报告损害与观察成本

比较任务成功、harm rate、unnecessary actions、inspect 次数、belief Brier/ECE、状态漂移恢复和 p95 延迟。环境 reset/snapshot、likelihood/阈值和动作 policy 都是可版本制品。


In [ ]:
def brier_danger(predicted, actual_danger):  # 定义本节可复用的核心函数。
    predicted, actual = np.asarray(predicted, dtype=float), np.asarray(actual_danger, dtype=float)  # 计算并保存当前步骤的中间状态。
    if predicted.shape != actual.shape or not np.isfinite(predicted).all() or not np.isfinite(actual).all():  # 按当前条件选择后续控制路径。
        raise ValueError("Brier 输入必须同形且有限")  # 遇到非法合同立即显式失败。
    if ((predicted < 0) | (predicted > 1) | (actual < 0) | (actual > 1)).any():  # 按当前条件选择后续控制路径。
        raise ValueError("Brier 输入必须位于 [0,1]")  # 遇到非法合同立即显式失败。
    return float(np.mean((predicted - actual) ** 2))  # 返回当前分支计算出的结果。

def artifact_hash(artifact):  # 定义本节可复用的核心函数。
    validate_policy(artifact)  # 执行当前语句以推进本节示例。
    return hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()  # 返回当前分支计算出的结果。

# 同一 observation 经 artifact 的 threshold/freshness 改变真实 loop 终态。
artifact = DEFAULT_POLICY  # 计算并保存当前步骤的中间状态。
loose = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.15, 2, 0.0)  # 计算并保存当前步骤的中间状态。
strict_threshold = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.08, 2, 0.0)  # 计算并保存当前步骤的中间状态。
strict_freshness = POMDPArtifact("env-v5", "sensor-likelihood-v3", "evidence-gated-eig-v3", 0.15, 1, 0.0)  # 计算并保存当前步骤的中间状态。
border_record = ObservationRecord("green", "sensor-a", 7, 8)  # 计算并保存当前步骤的中间状态。
loose_action, _, loose_trace = run_belief_agent(belief, [border_record], 1, 7, 10, loose)  # 计算并保存当前步骤的中间状态。
threshold_action, _, _ = run_belief_agent(belief, [border_record], 1, 7, 10, strict_threshold)  # 计算并保存当前步骤的中间状态。
freshness_action, freshness_posterior, freshness_trace = run_belief_agent(belief, [border_record], 1, 7, 10, strict_freshness)  # 计算并保存当前步骤的中间状态。
assert loose_action == "execute" and threshold_action == "abort" and freshness_action == "abort"  # 用受控断言验证关键不变量。
assert np.array_equal(freshness_posterior, belief)  # 用受控断言验证关键不变量。
assert any(event.get("status") == "stale" for event in freshness_trace)  # 用受控断言验证关键不变量。
digest = artifact_hash(artifact)  # 计算并保存当前步骤的中间状态。
assert brier_danger([0, 1], [0, 1]) == 0 and len(digest) == 64  # 用受控断言验证关键不变量。
assert digest != artifact_hash(POMDPArtifact("env-v5", "sensor-v4", artifact.action_policy, artifact.danger_threshold, artifact.freshness_limit, artifact.inspect_cost))  # 用受控断言验证关键不变量。
assert any(event.get("fresh_evidence") for event in loose_trace if event["state"] == "decide")  # 用受控断言验证关键不变量。


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
